# 08. 이용목적/차량구분/장애유형별 세분 분석

세그먼트별 접수건수, 취소율, 탑승전환율, 접수→승차 시간을 비교한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'

request_path = DATA_DIR / '서울시설공단_장애인콜택시 접수일시_월주차_파생컬럼_20251231.csv'
print(request_path)

usecols = [
    '접수일시', '승차일시', '취소일시',
    '이용목적', '차량구분', '장애유형', '접수시간대', '접수요일'
]

df_segment = pd.read_csv(request_path, usecols=usecols)
for col in ['접수일시', '승차일시', '취소일시']:
    df_segment[col] = pd.to_datetime(df_segment[col], errors='coerce')

df_segment = df_segment[
    (df_segment['접수일시'] >= '2025-01-01') &
    (df_segment['접수일시'] < '2026-01-01')
].copy()

df_segment['취소여부'] = df_segment['취소일시'].notna()
df_segment['탑승여부'] = df_segment['승차일시'].notna()
df_segment['접수_승차_분'] = (df_segment['승차일시'] - df_segment['접수일시']).dt.total_seconds() / 60
df_segment.loc[df_segment['접수_승차_분'] < 0, '접수_승차_분'] = pd.NA

print(f'분석 행 수: {len(df_segment):,}')
for col in ['접수일시', '승차일시', '취소일시']:
    print(f'{col} 결측/변환실패: {df_segment[col].isna().sum():,}건')

df_segment.head()


## 세그먼트 요약 함수


In [ ]:
def build_segment_summary(column_name, min_count=100):
    summary = (
        df_segment
        .dropna(subset=[column_name])
        .groupby(column_name)
        .agg(
            접수건수=('접수일시', 'size'),
            취소건수=('취소여부', 'sum'),
            탑승건수=('탑승여부', 'sum'),
            평균_접수승차분=('접수_승차_분', 'mean'),
            중앙값_접수승차분=('접수_승차_분', 'median'),
            p90_접수승차분=('접수_승차_분', lambda s: s.quantile(0.9)),
        )
        .reset_index()
    )
    summary['취소율(%)'] = summary['취소건수'] / summary['접수건수'] * 100
    summary['탑승전환율(%)'] = summary['탑승건수'] / summary['접수건수'] * 100
    return summary[summary['접수건수'] >= min_count].sort_values('접수건수', ascending=False).reset_index(drop=True)

purpose_demand_summary = build_segment_summary('이용목적')
vehicle_demand_summary = build_segment_summary('차량구분')
disability_demand_summary = build_segment_summary('장애유형')

segment_demand_summary = {
    '이용목적': purpose_demand_summary,
    '차량구분': vehicle_demand_summary,
    '장애유형': disability_demand_summary,
}

display(purpose_demand_summary)
display(vehicle_demand_summary)
display(disability_demand_summary)


## 이용목적별 접수건수와 취소율


In [ ]:
plot_data = purpose_demand_summary.sort_values('접수건수')
fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.barh(plot_data['이용목적'], plot_data['접수건수'], color='#4c78a8')
ax1.set_xlabel('접수건수')
ax1.set_ylabel('이용목적')
ax1.grid(axis='x', alpha=0.3)

ax2 = ax1.twiny()
ax2.plot(plot_data['취소율(%)'], plot_data['이용목적'], color='#e45756', marker='o', label='취소율')
ax2.set_xlabel('취소율(%)')

plt.title('이용목적별 접수건수와 취소율')
plt.tight_layout()
plt.show()


## 차량구분별 접수→승차 시간 분포


In [ ]:
vehicle_order = vehicle_demand_summary.sort_values('접수건수', ascending=False)['차량구분']
plot_df = df_segment[df_segment['차량구분'].isin(vehicle_order)].dropna(subset=['접수_승차_분']).copy()
plot_df['접수_승차_분_clip'] = plot_df['접수_승차_분'].clip(upper=180)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_df,
    x='차량구분',
    y='접수_승차_분_clip',
    order=vehicle_order
)
plt.title('차량구분별 접수→승차 시간 분포: 180분 초과는 180분으로 절단')
plt.xlabel('차량구분')
plt.ylabel('접수→승차 시간(분)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 장애유형별 접수건수와 취소율


In [ ]:
plot_data = disability_demand_summary.sort_values('접수건수')
fig, ax1 = plt.subplots(figsize=(12, 8))

ax1.barh(plot_data['장애유형'], plot_data['접수건수'], color='#4c78a8')
ax1.set_xlabel('접수건수')
ax1.set_ylabel('장애유형')
ax1.grid(axis='x', alpha=0.3)

ax2 = ax1.twiny()
ax2.plot(plot_data['취소율(%)'], plot_data['장애유형'], color='#e45756', marker='o', label='취소율')
ax2.set_xlabel('취소율(%)')

plt.title('장애유형별 접수건수와 취소율')
plt.tight_layout()
plt.show()


## 취소율/대기시간이 높은 세그먼트 후보


In [ ]:
for name, summary in segment_demand_summary.items():
    print(f'{name}별 취소율 상위')
    display(summary.sort_values('취소율(%)', ascending=False).head(10))
    print(f'{name}별 평균 접수→승차 시간 상위')
    display(summary.sort_values('평균_접수승차분', ascending=False).head(10))
